## Garbage collection & image scanning

Two operational concerns keep a registry healthy over time: reclaiming space, and knowing what's inside your images.

**Garbage collection.** A registry running for months accumulates thousands of blobs — layers for tags that were overwritten, or images nobody references. GC sweeps them. For `registry:2`:

```bash
docker exec registry registry garbage-collect --delete-untagged /etc/distribution/config.yml
```

It walks all manifests, walks all blobs those manifests reference, and **deletes any blob that isn't referenced**; `--delete-untagged` also removes manifests with no tag (the `<none>` images you see in `docker images`). Storage grows *surprisingly* fast — a CI pipeline pushing one image per commit can accumulate a multi-gigabyte registry in a quarter — so GC plus a **retention policy** ("keep the last 10 tags", "delete non-`prod-*` builds older than 30 days") is essential. On Harbor and the cloud registries you don't run GC by hand; you configure those retention rules and it happens for you.

**Image scanning.** Modern registries (Hub, ECR, GHCR, Harbor, Quay) scan pushed images against **CVE databases** and flag known vulnerabilities by severity. You can also scan locally with `trivy` or `grype`:

```bash
trivy image ghcr.io/myorg/api:1.2.3
# Total: 47 (LOW: 12, MEDIUM: 28, HIGH: 6, CRITICAL: 1)
```

Two common ways teams act on it: a **CI gate** that fails a build introducing a new `HIGH`/`CRITICAL` CVE, and **admission control** (Kyverno, OPA Gatekeeper) that refuses to schedule unscanned or vulnerable images in Kubernetes. Scanning is the **detection** half of supply-chain hygiene — it tells you *what's* vulnerable. **Reducing** the surface area — smaller bases, non-root users, distroless, dropped capabilities — is the other half, and it's exactly where module 08 goes next.